# Train Maddie — 3B or 7B (Colab)

Structure mirrors `config.py`, `data_loader.py`, `train.py`.
- **3B**: full fine-tune (fits T4 16GB with gradient checkpointing, small batch).
- **7B**: QLoRA (4-bit + LoRA) so it fits in 16GB; full 7B needs ~40GB VRAM.

## 1. Setup: Drive + GPU + pip

In [5]:
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
except Exception as e:
    DRIVE_MOUNTED = False
    print("Drive not mounted:", e)

get_ipython().system('pip install -q torch transformers datasets accelerate peft bitsandbytes')

# Disable safetensors auto-conversion thread (avoids JSONDecodeError in Colab)
import transformers.safetensors_conversion as _st_conv
_st_conv.auto_conversion = lambda *a, **k: None

# GPU required. Runtime → Change runtime type → GPU
get_ipython().system('nvidia-smi')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sun Mar  1 10:44:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|          

## 2. Config (like config.py)

In [6]:
# --- Model: 3B full fine-tune, or 7B QLoRA ---
MODEL_SIZE = "3b"   # "3b" | "7b"

if MODEL_SIZE == "3b":
    model_name = "openlm-research/open_llama_3b_v2"
    use_qlora = False
    block_size = 512   # 1024 if you have 24GB
elif MODEL_SIZE == "7b":
    model_name = "openlm-research/open_llama_7b_v2"
    use_qlora = True   # 4-bit + LoRA to fit 16GB
    block_size = 512
else:
    raise ValueError("MODEL_SIZE must be '3b' or '7b'")

tokenizer_use_fast = False  # Required for OpenLLaMA

# --- Dataset: streaming (like config.py) or HuggingFace preset ---
dataset_name = "mlfoundations/dclm-baseline-1.0"  # streaming
streaming_buffer_size = 50_000  # Colab: smaller buffer
use_streaming = True   # False = use preset below
preset_dataset = "wikitext"  # if use_streaming=False: wikitext | code_expert | alpaca | tiny_stories

# --- Training ---
max_steps = 5000      # Colab: keep low; increase if you have time
batch_size = 1
gradient_accumulation_steps = 8
learning_rate = 3e-4 if not use_qlora else 2e-4
warmup_steps = 200
logging_steps = 10
save_steps = 500
save_total_limit = 2

output_dir = "/content/drive/MyDrive/Maddie_3B" if DRIVE_MOUNTED else "./Maddie_3B"
if MODEL_SIZE == "7b":
    output_dir = "/content/drive/MyDrive/Maddie_7B" if DRIVE_MOUNTED else "./Maddie_7B"

use_bf16 = True
use_8bit_optimizer = True
max_grad_norm = 1.0
report_to = "none"

print(f"Model: {model_name}, QLoRA: {use_qlora}, block_size: {block_size}")
print(f"Output: {output_dir}")

Model: openlm-research/open_llama_3b_v2, QLoRA: False, block_size: 512
Output: /content/drive/MyDrive/Maddie_3B


## 3. Data loader (like data_loader.py)

In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch
from torch.utils.data import IterableDataset, Dataset

class StreamingDataset(IterableDataset):
    def __init__(self, dataset_name, tokenizer, buffer_size, block_size):
        self.ds = load_dataset(dataset_name, split="train", streaming=True)
        self.shuffled = self.ds.shuffle(buffer_size=buffer_size)
        self.tokenizer = tokenizer
        self.block_size = block_size

    def __iter__(self):
        pad_id = self.tokenizer.pad_token_id or self.tokenizer.eos_token_id
        for sample in self.shuffled:
            tok = self.tokenizer(
                sample["text"],
                truncation=True,
                max_length=self.block_size,
                padding="max_length",
                return_tensors=None,
            )
            input_ids = tok["input_ids"]
            attn = tok["attention_mask"]
            labels = [tid if attn[i] else -100 for i, tid in enumerate(input_ids)]
            yield {
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "attention_mask": torch.tensor(attn, dtype=torch.long),
            }

def get_preset_dataset(preset, tokenizer, block_size):
    if preset == "wikitext":
        data = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        text_col = "text"
    elif preset == "code_expert":
        data = load_dataset("HuggingFaceH4/CodeAlpaca_20k", split="train")
        data = data.map(lambda ex: {"text": f"Instruction: {ex['prompt']}\nResponse: {ex['completion']}"}, remove_columns=data.column_names)
        text_col = "text"
    elif preset == "alpaca":
        data = load_dataset("tatsu-lab/alpaca", split="train")
        data = data.map(lambda ex: {"text": f"Instruction: {ex['instruction']}\nInput: {ex.get('input') or ''}\nResponse: {ex['output']}"}, remove_columns=data.column_names)
        text_col = "text"
    else:
        data = load_dataset("roneneldan/TinyStories", split="train").select(range(30000))
        text_col = "text"
    def tokenize(examples):
        out = tokenizer(examples[text_col], truncation=True, max_length=block_size, padding="max_length")
        pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
        out["labels"] = [[id if id != pad_id else -100 for id in ids] for ids in out["input_ids"]]
        return out
    data = data.map(tokenize, batched=True, remove_columns=data.column_names)
    data.set_format("torch")
    return data

print("Data loader helpers defined.")

Data loader helpers defined.


## 4. Load tokenizer and dataset

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=tokenizer_use_fast)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if use_streaming:
    train_dataset = StreamingDataset(dataset_name, tokenizer, streaming_buffer_size, block_size)
    data_collator = None  # Trainer uses default; batches from iterable
else:
    train_dataset = get_preset_dataset(preset_dataset, tokenizer, block_size)
    data_collator = None

print("Dataset ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resolving data files:   0%|          | 0/27838 [00:00<?, ?it/s]

Dataset ready.


## 5. Load model and train (like train.py)

In [9]:
import torch

# Disable safetensors auto-conversion thread (causes JSONDecodeError in Colab)
import transformers.safetensors_conversion as _sc
_orig_auto = _sc.auto_conversion
def _noop_auto_conversion(*args, **kwargs):
    pass
_sc.auto_conversion = _noop_auto_conversion

from transformers import LlamaConfig, LlamaForCausalLM, Trainer, TrainingArguments, default_data_collator

cuda_ok = torch.cuda.is_available()
bf16_ok = cuda_ok and torch.cuda.is_bf16_supported()
use_bf16_actual = use_bf16 and bf16_ok
if not cuda_ok:
    raise RuntimeError("No GPU. Set Runtime → Change runtime type → GPU.")
print(f"GPU: {torch.cuda.get_device_name(0)}, bf16: {use_bf16_actual}")

torch.cuda.empty_cache()

if use_qlora:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
    model = LlamaForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto", use_safetensors=False)
    model = prepare_model_for_kbit_training(model)
    peft_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(model, peft_config)
else:
    config = LlamaConfig.from_pretrained(model_name)
    dtype = torch.bfloat16 if use_bf16_actual else torch.float16
    model = LlamaForCausalLM.from_pretrained(model_name, config=config, use_safetensors=False).to(dtype)
    model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    max_steps=max_steps,
    learning_rate=learning_rate,
    warmup_steps=warmup_steps,
    logging_steps=logging_steps,
    save_steps=save_steps,
    save_total_limit=save_total_limit,
    bf16=use_bf16_actual,
    fp16=not use_bf16_actual,
    remove_unused_columns=False,
    report_to=report_to,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    max_grad_norm=max_grad_norm,
    optim="adamw_bnb_8bit" if use_8bit_optimizer else "adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
)

trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Done. Saved to", output_dir)

GPU: Tesla T4, bf16: True


Loading weights:   0%|          | 0/237 [00:00<?, ?it/s]

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'